# BrewBeats — Mini Project 2: Section B (SQL on DatabricksSQL)
**File:** `Section_B_Q3.ipynb`  
**Folder:** `Section B - SQL on DatabricksSQL/`  
**Stack:** Databricks SQL / Spark SQL

---
## Section B - Q3: Write a query using a CTE to find total revenue per month.

In [1]:
# === Section B - Q3: Total revenue per month using a CTE (Databricks SQL) ===
import warnings
warnings.filterwarnings("ignore")  # Silence Python FutureWarnings so output stays clean
from pyspark.sql import SparkSession

# 1. Start our Spark session to run Databricks / Spark SQL queries
spark = (
    SparkSession.builder
    .appName("BrewBeats_SectionB_Q3")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# 2. Load 'coffee_sales.csv' and register it as a temporary SQL table named 'coffee_sales'
coffee_df = spark.read.csv("coffee_sales.csv", header=True, inferSchema=True)
coffee_df.createOrReplaceTempView("coffee_sales")

# 3. Write our Databricks SQL query using a CTE (Common Table Expression):
#    - WITH monthly_sales AS (...): defines a temporary helper table (CTE) that extracts the month columns
#      ('year_month', 'month_number', 'month_name') from 'date' first, making our query clean and modular.
#    - Main SELECT ... FROM monthly_sales: groups by those month columns, counts total transactions,
#      and sums 'money' (ROUND(SUM(money), 2)) to calculate the total revenue per month.
#    - ORDER BY year_month ASC: sorts chronologically from 2024-03 to 2025-03.
q3_query = """
WITH monthly_sales AS (
    SELECT
        DATE_FORMAT(date, 'yyyy-MM') AS year_month,
        MONTH(date) AS month_number,
        DATE_FORMAT(date, 'MMMM') AS month_name,
        money
    FROM coffee_sales
)
SELECT
    year_month,
    month_number,
    month_name,
    COUNT(*) AS total_transactions,
    ROUND(SUM(money), 2) AS monthly_revenue
FROM monthly_sales
GROUP BY year_month, month_number, month_name
ORDER BY year_month ASC
"""

print("=== Section B - Q3: Total Revenue per Month Using a CTE ===")
spark.sql(q3_query).show(20, truncate=False)

=== Section B - Q3: Total Revenue per Month Using a CTE ===
+----------+------------+----------+------------------+---------------+
|year_month|month_number|month_name|total_transactions|monthly_revenue|
+----------+------------+----------+------------------+---------------+
|   2024-03|           3|     March|               206|         7050.2|
|   2024-04|           4|     April|               196|        6720.56|
|   2024-05|           5|       May|               267|        9063.42|
|   2024-06|           6|      June|               227|        7758.76|
|   2024-07|           7|      July|               237|        6915.94|
|   2024-08|           8|    August|               272|        7613.84|
|   2024-09|           9| September|               344|        9988.64|
|   2024-10|          10|   October|               426|       13891.16|
|   2024-11|          11|  November|               259|        8590.54|
|   2024-12|          12|  December|               259|        8237.74|
|   